# Kronos RAG (extracted from the development notebook)

Reference notebook for the clean scripts in this repo (`main.py`, `model.py`, `transfer.py`, `fetch_data.py`, `dataset.py`, `rag.py`).

Dependencies are in `requirements.txt`:

```bash
pip install -r requirements.txt
```

The weight-transfer cell expects the upstream Kronos repo cloned next to this one (`git clone https://github.com/shiyu-coder/Kronos`).


In [ ]:
import argparse
import os
import pickle
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from array_record.python.array_record_module import ArrayRecordWriter

NY_TZ = ZoneInfo("America/New_York")

FEATURES = ("open", "high", "low", "close", "vol", "amt")

SP_TICKERS = [
    'A', 'AAL', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI',
    'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG',
    'AKAM', 'ALB', 'ALGN', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN',
    'AMP', 'AMT', 'AMZN', 'ANET', 'ANSS', 'AON', 'AOS', 'APA', 'APD', 'APH',
    'APO', 'APTV', 'ARE', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP',
    'AZO', 'BA', 'BAC', 'BALL', 'BAX', 'BBWI', 'BBY', 'BDX', 'BEN', 'BF.B',
    'BG', 'BIIB', 'BK', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK.B',
    'BRO', 'BSX', 'BWA', 'BX', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CAT',
    'CB', 'CBOE', 'CBRE', 'CCI', 'CCJ', 'CDNS', 'CDW', 'CE', 'CEG', 'CF',
    'CFG', 'CHD', 'CHRW', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CMA', 'CMCSA',
    'CME', 'CMG', 'CMI', 'CMS', 'CNC', 'CNP', 'COF', 'COO', 'COP', 'COR',
    'COST', 'CPAY', 'CPB', 'CPRT', 'CPT', 'CRL', 'CRM', 'CRWD', 'CSGP', 'CSX',
    'CTAS', 'CTRA', 'CTSH', 'CTVA', 'CVS', 'CVX', 'CZR', 'D', 'DAL', 'DD',
    'DE', 'DELL', 'DFS', 'DG', 'DGX', 'DHI', 'DHR', 'DIS', 'DLR',
    'DLTR', 'DOC', 'DOV', 'DOW', 'DPZ', 'DRI', 'DTE', 'DUK', 'DVA', 'DVN',
    'DXCM', 'EA', 'EBAY', 'ECL', 'ED', 'EFX', 'EG', 'EIX', 'EL', 'ELV',
    'EMR', 'ENPH', 'EOG', 'EPAM', 'EQIX', 'EQR', 'EQT', 'ERIE', 'ES', 'ESS',
    'ETN', 'ETR', 'ETSY', 'EVRG', 'EW', 'EXC', 'EXPD', 'EXPE', 'EXR',
    'F', 'FANG', 'FAST', 'FCX', 'FDS', 'FDX', 'FE', 'FFIV', 'FI', 'FICO',
    'FIS', 'FITB', 'FMC', 'FOX', 'FOXA', 'FRT', 'FSLR', 'FTNT', 'FTV', 'GD',
    'GDDY', 'GE', 'GEHC', 'GEV', 'GEN', 'GILD', 'GIS', 'GL', 'GLW', 'GM',
    'GNRC', 'GOOG', 'GOOGL', 'GPC', 'GPN', 'GRMN', 'GS', 'GWW', 'HAL', 'HAS',
    'HBAN', 'HCA', 'HD', 'HIG', 'HII', 'HLT', 'HOLX', 'HON', 'HPE',
    'HPQ', 'HRL', 'HSIC', 'HST', 'HSY', 'HUBB', 'HUM', 'HWM', 'IBM', 'ICE',
    'IDXX', 'IEX', 'IFF', 'INCY', 'INSP', 'INTC', 'INTU', 'INVH', 'IP', 'IPG',
    'IQV', 'IR', 'IRM', 'ISRG', 'IT', 'ITW', 'IVZ', 'J', 'JBHT', 'JBL',
    'JCI', 'JKHY', 'JNJ', 'JNPR', 'JPM', 'K', 'KDP', 'KEY', 'KEYS', 'KHC',
    'KIM', 'KLAC', 'KMB', 'KMI', 'KMX', 'KO', 'KR', 'KVUE', 'L', 'LDOS',
    'LEN', 'LH', 'LHX', 'LRCX', 'LNC', 'LNT', 'LOW', 'LULU', 'LUV',
    'LVS', 'LW', 'LYB', 'LYV', 'MA', 'MAA', 'MAR', 'MAS', 'MCD', 'MCHP',
    'MCK', 'MCO', 'MDLZ', 'MDT', 'MET', 'META', 'MGM', 'MHK', 'MKC', 'MKTX',
    'MLM', 'MMC', 'MMM', 'MNST', 'MO', 'MOH', 'MOS', 'MPC', 'MPWR', 'MRNA',
    'MS', 'MSCI', 'MSFT', 'MSI', 'MTB', 'MTD', 'MU', 'NCLH', 'NDAQ', 'NDSN',
    'NEE', 'NEM', 'NFLX', 'NI', 'NKE', 'NOC', 'NOW', 'NRG', 'NSC',
    'NTAP', 'NTRS', 'NUE', 'NVDA', 'NVR', 'NWS', 'NWSA', 'NXPI', 'O', 'ODFL',
    'OKE', 'OMC', 'ON', 'ORLY', 'ORCL', 'OTIS', 'OXY', 'PANW', 'PARA', 'PAYX',
    'PAYC', 'PYPL', 'PCAR', 'PCG', 'PEG', 'PEP', 'PFE', 'PFG', 'PG', 'PGR',
    'PH', 'PHM', 'PKG', 'PLD', 'PLTR', 'PM', 'PNC', 'PNR', 'PNW', 'PODD',
    'POOL', 'PPG', 'PPL', 'PRU', 'PSA', 'PSX', 'PTC', 'PWR', 'QCOM',
    'QRVO', 'RCL', 'REG', 'REGN', 'RF', 'RJF', 'RL', 'RMD', 'ROK', 'ROL',
    'ROP', 'ROST', 'RSG', 'RVTY', 'SBAC', 'SBUX', 'SCHW', 'SHW', 'SJM', 'SLB',
    'SMCI', 'SNA', 'SNPS', 'SO', 'SPG', 'SPGI', 'SRE', 'STE', 'STLD', 'STT',
    'STX', 'STZ', 'SWK', 'SWKS', 'SYK', 'SYF', 'SYY', 'T', 'TAP', 'TDG',
    'TDY', 'TECH', 'TEL', 'TER', 'TFC', 'TFX', 'TGT', 'TJX', 'TMO', 'TMUS',
    'TPR', 'TRGP', 'TRMB', 'TROW', 'TRV', 'TSCO', 'TSLA', 'TSN', 'TYL', 'UA',
    'UAA', 'UAL', 'UDR', 'UHS', 'ULTA', 'UNH', 'UNP', 'UPS', 'URI', 'USB',
    'V', 'VICI', 'VLO', 'VLTO', 'VMC', 'VRSK', 'VRSN', 'VRTX', 'VST', 'VTR',
    'VTRS', 'VZ', 'WAB', 'WAT', 'WBA', 'WBD', 'WDC', 'WEC', 'WELL', 'WFC',
    'WM', 'WMB', 'WMT', 'WRB', 'WRK', 'WST', 'WTW', 'WY', 'WYNN', 'XEL',
    'XOM', 'XYL', 'YUM', 'ZBH', 'ZBRA', 'ZTS'
]

TRAIN_FRAC = 0.80
ALPACA_API_KEY    = os.environ["ALPACA_API_KEY"]
ALPACA_SECRET_KEY = os.environ["ALPACA_SECRET_KEY"]

def stamps_from_index(index):
    idx = pd.DatetimeIndex(index)
    return np.stack([
        idx.minute.values,
        idx.hour.values,
        idx.weekday.values,
        idx.day.values,
        idx.month.values,
    ], axis=-1).astype(np.int32)


def fetch_tickers(symbols, start, end, chunk=100):
    from alpaca.data.historical import StockHistoricalDataClient
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.timeframe import TimeFrame
    from alpaca.data.enums import DataFeed, Adjustment

    client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

    frames = {}
    for i in range(0, len(symbols), chunk):
        batch = symbols[i:i + chunk]
        req = StockBarsRequest(
            symbol_or_symbols=batch,
            timeframe=TimeFrame.Minute,
            start=start,
            end=end,
            feed="iex",
            adjustment=Adjustment.ALL
        )
        df = client.get_stock_bars(req).df
        for sym in batch:
            if isinstance(df.index, pd.MultiIndex):
                if sym not in df.index.get_level_values("symbol"):
                    continue
                sub = df.xs(sym, level="symbol").sort_index()
            else:
                sub = df.sort_index()

            sub.index = pd.DatetimeIndex(sub.index).tz_convert(NY_TZ).tz_localize(None)
            out = pd.DataFrame(index=sub.index)
            out["open"], out["high"] = sub["open"], sub["high"]
            out["low"], out["close"] = sub["low"], sub["close"]
            out["vol"] = sub["volume"]
            vwap = sub["vwap"] 
            out["amt"] = sub["volume"] * vwap
            out = out.interpolate(limit_direction="both")
            frames[sym] = out
        print(f"[fetch] {min(i + chunk, len(symbols))}/{len(symbols)} symbols completed")
    return frames


def write_windows_arrayrecord(frames, train_frac, lookback, predict, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    train_writer = ArrayRecordWriter(os.path.join(output_dir, "train.arrayrecord"), options="group_size:1")
    val_writer = ArrayRecordWriter(os.path.join(output_dir, "val.arrayrecord"), options="group_size:1")

    train_count, val_count = 0, 0
    window_len = lookback + predict

    for symbol, df in frames.items():
        series = df[list(FEATURES)].to_numpy(dtype=np.float32)
        stamps = stamps_from_index(df.index)
        n = len(series)

        if n < window_len:
            continue

        split_idx = int(n * train_frac)

        for i in range(0, n - window_len + 1, window_len):
            x_win = series[i : i + window_len].copy()
            st_win = stamps[i : i + window_len].copy()

            past = x_win[:lookback]
            mean = np.mean(past, axis=0, keepdims=True)
            std = np.std(past, axis=0, keepdims=True)
            x_norm = np.clip((x_win - mean) / (std + 1e-5), -CLIP, CLIP).astype(np.float32)

            record = {
                "symbol": symbol,
                "x": x_norm,
                "stamps": st_win,
            }
            serialized = pickle.dumps(record)

            if (i + window_len) <= split_idx:
                train_writer.write(serialized)
                train_count += 1
            else:
                val_writer.write(serialized)
                val_count += 1

    train_writer.close()
    val_writer.close()
    print(f"[write] Wrote {train_count} train records and {val_count} val records.")


YEARS = 2
LOOKBACK = 90
PREDICT = 10
WINDOW = LOOKBACK + PREDICT
CLIP = 5.0

end = datetime.now(NY_TZ)
start = end - timedelta(days=int(YEARS * 365.25))

frames = fetch_tickers(SP_TICKERS, start, end)
write_windows_arrayrecord(frames, TRAIN_FRAC, LOOKBACK, PREDICT, "splits")

In [ ]:
import os
import pickle

import grain
import numpy as np
from array_record.python.array_record_module import ArrayRecordReader


BATCH_SIZE=64
NUM_WORKERS=0


class ParsePickleRecord(grain.transforms.Map):
    def map(self, record_bytes: bytes):
        return pickle.loads(record_bytes)

def make_dataloaders(train_arrayrecord_path, val_arrayrecord_path):
    data_source = grain.sources.ArrayRecordDataSource(train_arrayrecord_path)

    transformations = [
        ParsePickleRecord(),  
        grain.transforms.Batch(batch_size=BATCH_SIZE, drop_remainder=True)
    ]

    sampler = grain.samplers.IndexSampler(
        num_records=len(data_source),
        shuffle=True,
        seed=42,
        num_epochs=1,
    )

    train_dataloader = grain.DataLoader(
        data_source=data_source,
        sampler=sampler,
        operations=transformations,  
        worker_count=NUM_WORKERS,     
    )

    data_source = grain.sources.ArrayRecordDataSource(val_arrayrecord_path)

    transformations = [
        ParsePickleRecord(),
        grain.transforms.Batch(batch_size=1, drop_remainder=True)
    ]

    sampler = grain.samplers.IndexSampler(
        num_records=len(data_source),
        shuffle=False,
        num_epochs=1,
    )

    val_dataloader = grain.DataLoader(
        data_source=data_source,
        sampler=sampler,
        operations=transformations,  
        worker_count=NUM_WORKERS,     
    )

    return train_dataloader, val_dataloader

train, val = make_dataloaders("splits/train.arrayrecord", "splits/val.arrayrecord")


In [ ]:
import jax
import jax.numpy as jnp
import optax
from einshape import jax_einshape as einshape
from flax import nnx


class BinarySphericalQuantizer(nnx.Module):
    def __init__(self, embed_dim, beta, gamma0, gamma, zeta, group_size, inv_temperature=1.0, l2_norm=True, rngs=nnx.Rngs(0)):
        self.embed_dim = embed_dim
        self.l2_norm = l2_norm
        self.beta = beta
        self.gamma0 = gamma0
        self.gamma = gamma
        self.zeta = zeta
        self.group_size = group_size
        self.inv_temperature = inv_temperature

        assert embed_dim % group_size == 0, f"embed_dim ({embed_dim}) must be divisible by group_size ({group_size})"
        self.num_groups = embed_dim // group_size

        # Precompute bit conversion bases
        self.basis = nnx.Variable(2 ** jnp.arange(embed_dim - 1, -1, -1, dtype=jnp.int32))
        self.group_basis = nnx.Variable(2 ** jnp.arange(group_size - 1, -1, -1, dtype=jnp.int32))

        # Precompute group codebook table of size (2^group_size, group_size) with values in {-1.0, +1.0}
        group_codes = jnp.arange(2 ** group_size)
        self.group_codebook = nnx.Variable((group_codes[:, None] // self.group_basis) % 2 * 2.0 - 1.0)

    def quantize(self, z):
        zhat = jnp.where(z > 0, 1.0, -1.0)
        return z + jax.lax.stop_gradient(zhat - z)

    def codes_to_indexes(self, zhat):
        zb = ((zhat > 0).astype(jnp.int32))
        return jnp.sum(zb * self.basis, axis=-1)

    def codes_to_group_indexes(self, zhat):
        zhat_groups = zhat.reshape(*zhat.shape[:-1], self.num_groups, self.group_size)
        zb_groups = ((zhat_groups > 0).astype(jnp.int32))
        return jnp.sum(zb_groups * self.group_basis, axis=-1)

    def soft_entropy_loss(self, z):
        scale = 1.0 / jnp.sqrt(self.embed_dim) if self.l2_norm else 1.0
        group_cb_norm = self.group_codebook * scale
        divided_z = z.reshape(*z.shape[:-1], self.num_groups, self.group_size)

        # Distance and softmax probabilities over group codebook entries
        distance = -2.0 * jnp.einsum('...gc,dc->...gd', divided_z, group_cb_norm)
        prob = jax.nn.softmax(-distance * self.inv_temperature, axis=-1)

        # Analytical per-sample entropy
        p = jax.nn.sigmoid(-4.0 * z * scale * self.inv_temperature)
        per_sample_entropy = -jnp.sum(p * jnp.log(p + 1e-8) + (1.0 - p) * jnp.log(1.0 - p + 1e-8), axis=-1).mean()

        # Average probability across batch and time dimensions
        reduce_axes = tuple(range(prob.ndim - 2))
        avg_prob = jnp.mean(prob, axis=reduce_axes)  # Shape: (num_groups, 2^group_size)

        # Codebook entropy H
        codebook_entropy = -jnp.sum(avg_prob * jnp.log(avg_prob + 1e-8), axis=-1).sum()

        return per_sample_entropy, codebook_entropy, avg_prob

    def __call__(self, z, collect_metrics=True):
        zq = self.quantize(z)
        q_scale = 1.0 / jnp.sqrt(self.embed_dim) if self.l2_norm else 1.0
        zq = zq * q_scale

        if not collect_metrics:
            return zq, jnp.array(0.0), {}

        commit_loss = self.beta * jnp.mean(jnp.sum((jax.lax.stop_gradient(zq) - z) ** 2, axis=-1))

        per_sample_entropy, cb_entropy, avg_prob = self.soft_entropy_loss(z)
        entropy_penalty = self.gamma0 * per_sample_entropy - self.gamma * cb_entropy

        total_bsq_loss = commit_loss + self.zeta * entropy_penalty / self.inv_temperature

        zq_detached = jax.lax.stop_gradient(zq)
        indices = self.codes_to_indexes(zq_detached)
        group_indices = self.codes_to_group_indexes(zq_detached)

        metrics = {
            "H": cb_entropy,
            "per_sample_entropy": per_sample_entropy,
            "commit_loss": commit_loss,
            "indices": indices,
            "group_indices": group_indices,
            "avg_prob": avg_prob
        }

        return zq, total_bsq_loss, metrics


class BSQuantizer(nnx.Module):
    def __init__(self, s1_bits, s2_bits, beta, gamma0, gamma, zeta, group_size, rngs=nnx.Rngs(0)):
        self.s1_bits = s1_bits
        self.s2_bits = s2_bits
        self.codebook_dim = s1_bits + s2_bits
        self.bsq = BinarySphericalQuantizer(self.codebook_dim, beta, gamma0, gamma, zeta, group_size, rngs=rngs)

    def bits_to_indices(self, bits):
        set_bits = (bits >= 0).astype(jnp.int32)
        powers = 2 ** jnp.arange(bits.shape[-1], dtype=jnp.int32)
        return jnp.einsum('...d,d->...', set_bits, powers)

    def __call__(self, z, half=False, collect_metrics=True):
        norm = jnp.linalg.norm(z, axis=-1, keepdims=True)
        z_norm = z / jnp.maximum(norm, 1e-12)
        quantized, bsq_loss, metrics = self.bsq(z_norm, collect_metrics=collect_metrics)
        if half:
            q_pre = quantized[:, :, :self.s1_bits]
            q_post = quantized[:, :, self.s1_bits:]
            z_indices = [self.bits_to_indices(q_pre), self.bits_to_indices(q_post)]
        else:
            z_indices = self.bits_to_indices(quantized)
        return bsq_loss, quantized, z_indices, metrics


def make_linear(in_features, out_features, lora_rank=0, use_bias=True, rngs=nnx.Rngs(0)):
    if lora_rank > 0:
        return nnx.LoRALinear(in_features, out_features, lora_rank=lora_rank, use_bias=use_bias, rngs=rngs)
    else:
        return nnx.Linear(in_features, out_features, use_bias=use_bias, rngs=rngs)


class FeedForward(nnx.Module):
    def __init__(self, d_model, ff_dim, ffn_dropout_p=0.0, lora_rank=0, rngs=nnx.Rngs(0)):
        self.w1 = make_linear(d_model, ff_dim, lora_rank=lora_rank, use_bias=False, rngs=rngs)
        self.w3 = make_linear(d_model, ff_dim, lora_rank=lora_rank, use_bias=False, rngs=rngs)
        self.w2 = make_linear(ff_dim, d_model, lora_rank=lora_rank, use_bias=False, rngs=rngs)
        self.ffn_dropout = nnx.Dropout(ffn_dropout_p, rngs=rngs)

    def __call__(self, x):
        h = nnx.silu(self.w1(x)) * self.w3(x)
        return self.ffn_dropout(self.w2(h))


class RotaryPositionalEmbedding(nnx.Module):
    def __init__(self, dim):
        self.dim = dim
        self.inv_freq = nnx.Variable(1.0 / (10000.0 ** (jnp.arange(0, dim, 2, dtype=jnp.float32) / dim)))

    def _rotate_half(self, x):
        x1, x2 = jnp.split(x, 2, axis=-1)
        return jnp.concatenate([-x2, x1], axis=-1)

    def _get_embed(self, seq_len):
        t = jnp.arange(seq_len, dtype=jnp.float32)
        freqs = jnp.einsum('i,j->ij', t, self.inv_freq)
        emb = jnp.concatenate([freqs, freqs], axis=-1)
        cos = einshape('sd->11sd', jnp.cos(emb))
        sin = einshape('sd->11sd', jnp.sin(emb))
        return cos, sin

    def __call__(self, q, k):
        q_len = q.shape[-2]
        k_len = k.shape[-2]

        q_cos, q_sin = self._get_embed(q_len)
        q_out = q * q_cos + self._rotate_half(q) * q_sin

        if q_len == k_len:
            k_out = k * q_cos + self._rotate_half(k) * q_sin
        else:
            k_cos, k_sin = self._get_embed(k_len)
            k_out = k * k_cos + self._rotate_half(k) * k_sin

        return q_out, k_out


class MultiHeadAttentionWithRoPE(nnx.Module):
    def __init__(self, d_model, n_heads, attn_dropout_p, resid_dropout_p, lora_rank, rngs=nnx.Rngs(0)):
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.k_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.v_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.out_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nnx.Dropout(attn_dropout_p, rngs=rngs)
        self.resid_dropout = nnx.Dropout(resid_dropout_p, rngs=rngs)

    def __call__(self, x, key_padding_mask=None):
        split_heads = lambda t: einshape('bs(hd)->bhsd', t, h=self.n_heads)
        q = split_heads(self.q_proj(x))
        k = split_heads(self.k_proj(x))
        v = split_heads(self.v_proj(x))

        q, k = self.rotary(q, k)
        scores = jnp.einsum('bhqd,bhkd->bhqk', q, k) / jnp.sqrt(self.head_dim)

        seq_len = x.shape[1]
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        scores = jnp.where(einshape('qk->11qk', causal_mask), scores, -jnp.inf)

        if key_padding_mask is not None:
            scores = jnp.where(einshape('bk->b11k', key_padding_mask.astype(jnp.bool_)), scores, -jnp.inf)

        attn_weights = nnx.softmax(scores, axis=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = jnp.einsum('bhqk,bhkd->bhqd', attn_weights, v)
        attn_output = einshape('bhsd->bs(hd)', attn_output)
        return self.resid_dropout(self.out_proj(attn_output))


class MultiHeadCrossAttentionWithRoPE(nnx.Module):
    def __init__(self, d_model, n_heads, attn_dropout_p, resid_dropout_p, lora_rank, deterministic=False, rngs=nnx.Rngs(0)):
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.deterministic = deterministic
        self.q_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.k_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.v_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.out_proj = make_linear(d_model, d_model, lora_rank=lora_rank, rngs=rngs)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nnx.Dropout(attn_dropout_p, rngs=rngs)
        self.resid_dropout = nnx.Dropout(resid_dropout_p, rngs=rngs)

    def __call__(self, query, key, value, key_padding_mask=None):
        q_len = query.shape[1]
        seq_len = key.shape[1]
        q = einshape('bq(hd)->bhqd', self.q_proj(query), h=self.n_heads)
        k = einshape('bk(hd)->bhkd', self.k_proj(key), h=self.n_heads)
        v = einshape('bk(hd)->bhkd', self.v_proj(value), h=self.n_heads)

        q, k = self.rotary(q, k)
        scores = jnp.einsum('bhqd,bhkd->bhqk', q, k) / jnp.sqrt(self.head_dim)
        
        if not self.deterministic:
            causal_mask = jnp.tril(jnp.ones((q_len, seq_len), dtype=jnp.bool_))
            scores = jnp.where(einshape('qk->11qk', causal_mask), scores, -jnp.inf)

        if key_padding_mask is not None:
            scores = jnp.where(einshape('bk->b11k', key_padding_mask), scores, -jnp.inf)

        attn_weights = nnx.softmax(scores, axis=-1)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = jnp.einsum('bhqk,bhkd->bhqd', attn_weights, v)
        attn_output = einshape('bhqd->bq(hd)', attn_output)
        return self.resid_dropout(self.out_proj(attn_output))



class HierarchicalEmbedding(nnx.Module):
    def __init__(self, s1_bits, s2_bits, d_model, rngs=nnx.Rngs(0)):
        self.s1_bits = s1_bits
        self.s2_bits = s2_bits
        self.d_model = d_model

        self.emb_s1 = nnx.Embed(2 ** s1_bits, d_model, rngs=rngs)
        self.emb_s2 = nnx.Embed(2 ** s2_bits, d_model, rngs=rngs)
        self.fusion_proj = nnx.Linear(d_model * 2, d_model, rngs=rngs)


    def __call__(self, token_ids):
        s1_ids, s2_ids = token_ids
        s1_emb = self.emb_s1(s1_ids) * jnp.sqrt(self.d_model)
        s2_emb = self.emb_s2(s2_ids) * jnp.sqrt(self.d_model)
        return self.fusion_proj(jnp.concatenate([s1_emb, s2_emb], axis=-1))


class DependencyAwareLayer(nnx.Module):
    def __init__(self, d_model, n_heads=4, attn_dropout_p=0.0, resid_dropout_p=0.0, lora_rank=0, rngs=nnx.Rngs(0)):
        self.cross_attn = MultiHeadCrossAttentionWithRoPE(d_model, n_heads, attn_dropout_p, resid_dropout_p, lora_rank=lora_rank, rngs=rngs)
        self.norm = nnx.RMSNorm(d_model, epsilon=1e-5, rngs=rngs)

    def __call__(self, hidden_states, sibling_embed, key_padding_mask=None):
        attn_out = self.cross_attn(
            query=sibling_embed,
            key=hidden_states,
            value=hidden_states,
            key_padding_mask=key_padding_mask,
        )
        return self.norm(hidden_states + attn_out)


class TransformerBlock(nnx.Module):
    def __init__(self, d_model, n_heads, ff_dim, ffn_dropout_p, attn_dropout_p, resid_dropout_p, lora_rank, rngs=nnx.Rngs(0)):
        self.norm1 = nnx.RMSNorm(d_model, epsilon=1e-5, rngs=rngs)
        self.self_attn = MultiHeadAttentionWithRoPE(d_model, n_heads, attn_dropout_p, resid_dropout_p, lora_rank=lora_rank, rngs=rngs)
        self.norm2 = nnx.RMSNorm(d_model, epsilon=1e-5, rngs=rngs)
        self.ffn = FeedForward(d_model, ff_dim, ffn_dropout_p, lora_rank=lora_rank, rngs=rngs)

    def __call__(self, x, key_padding_mask=None):
        residual = x
        x_norm = self.norm1(x)
        attn_out = self.self_attn(x_norm, key_padding_mask=key_padding_mask)
        x = residual + attn_out

        residual = x
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        return residual + ffn_out


class DualHead(nnx.Module):
    def __init__(self, s1_bits, s2_bits, d_model, rngs=nnx.Rngs(0)):
        self.proj_s1 = nnx.Linear(d_model, 2 ** s1_bits, rngs=rngs)
        self.proj_s2 = nnx.Linear(d_model, 2 ** s2_bits, rngs=rngs)

    def __call__(self, x):
        return self.proj_s1(x)

    def cond_forward(self, x2):
        return self.proj_s2(x2)


class TemporalEmbedding(nnx.Module):
    def __init__(self, d_model, rngs=nnx.Rngs(0)):
        minute_size = 60
        hour_size = 24
        weekday_size = 7
        day_size = 32
        month_size = 13

        self.minute_embed = nnx.Embed(minute_size, d_model, rngs=rngs)
        self.hour_embed = nnx.Embed(hour_size, d_model, rngs=rngs)
        self.weekday_embed = nnx.Embed(weekday_size, d_model, rngs=rngs)
        self.day_embed = nnx.Embed(day_size, d_model, rngs=rngs)
        self.month_embed = nnx.Embed(month_size, d_model, rngs=rngs)

    def __call__(self, x):
        x = x.astype(jnp.int32)
        minute_x = self.minute_embed(x[:, :, 0])
        hour_x = self.hour_embed(x[:, :, 1])
        weekday_x = self.weekday_embed(x[:, :, 2])
        day_x = self.day_embed(x[:, :, 3])
        month_x = self.month_embed(x[:, :, 4])
        return hour_x + weekday_x + day_x + month_x + minute_x


class KronosTokenizer(nnx.Module):
    def __init__(self, config, rngs=nnx.Rngs(0)):
        self.d_in = config["d_in"]
        self.d_model = config["d_model"]
        self.n_heads = config["n_heads"]
        self.ff_dim = config["ff_dim"]
        self.enc_layers = config["n_enc_layers"]
        self.dec_layers = config["n_dec_layers"]
        self.ffn_dropout_p = config["ffn_dropout_p"]
        self.attn_dropout_p = config["attn_dropout_p"]
        self.resid_dropout_p = config["resid_dropout_p"]
        self.lora_rank = config["lora_rank"]
        self.s1_bits = config["s1_bits"]
        self.s2_bits = config["s2_bits"]
        self.beta = config["beta"]
        self.gamma0 = config["gamma0"]
        self.gamma = config["gamma"]
        self.zeta = config["zeta"]
        self.group_size = config["group_size"]

        self.codebook_dim = self.s1_bits + self.s2_bits

        self.embed = make_linear(self.d_in, self.d_model, lora_rank=self.lora_rank, rngs=rngs)
        self.head = make_linear(self.d_model, self.d_in, lora_rank=self.lora_rank, rngs=rngs)

        self.encoder = nnx.List([
            TransformerBlock(self.d_model, self.n_heads, self.ff_dim, self.ffn_dropout_p, self.attn_dropout_p, self.resid_dropout_p, self.lora_rank, rngs=rngs)
            for _ in range(self.enc_layers - 1)
        ])
        self.decoder = nnx.List([
            TransformerBlock(self.d_model, self.n_heads, self.ff_dim, self.ffn_dropout_p, self.attn_dropout_p, self.resid_dropout_p, self.lora_rank, rngs=rngs)
            for _ in range(self.dec_layers - 1)
        ])

        self.quant_embed = make_linear(self.d_model, self.codebook_dim, lora_rank=self.lora_rank, rngs=rngs)
        self.post_quant_embed_pre = make_linear(self.s1_bits, self.d_model, lora_rank=self.lora_rank, rngs=rngs)
        self.post_quant_embed = make_linear(self.codebook_dim, self.d_model, lora_rank=self.lora_rank, rngs=rngs)
        self.tokenizer = BSQuantizer(self.s1_bits, self.s2_bits, self.beta, self.gamma0, self.gamma, self.zeta, self.group_size, rngs=rngs)

    def indices_to_bits(self, x, half=False):
        if half:
            x1, x2 = x[0], x[1]
            half_dim = self.codebook_dim // 2
            mask = 2 ** jnp.arange(half_dim, dtype=jnp.int32)
            b1 = (einshape('bs->bs1', x1.astype(jnp.int32)) & mask) != 0
            b2 = (einshape('bs->bs1', x2.astype(jnp.int32)) & mask) != 0
            bits = jnp.concatenate([b1, b2], axis=-1)
        else:
            mask = 2 ** jnp.arange(self.codebook_dim, dtype=jnp.int32)
            bits = (einshape('bs->bs1', x.astype(jnp.int32)) & mask) != 0

        bits = bits.astype(jnp.float32) * 2 - 1 
        q_scale = 1.0 / (self.codebook_dim ** 0.5)
        return bits * q_scale

    def decode(self, x, half=True):
        quantized = self.indices_to_bits(x, half=half)
        z = self.post_quant_embed(quantized)
        for layer in self.decoder:
            z = layer(z)
        return self.head(z)

    def encode(self, x, half=True, collect_metrics=False):
        z = self.embed(x)
        for layer in self.encoder:
            z = layer(z)
        z = self.quant_embed(z)
        _, _, z_indices, _ = self.tokenizer(z, half=half, collect_metrics=collect_metrics)
        return z_indices[0], z_indices[1]
    
    def compute_loss(self, x, z_pre, z, bsq_loss):
        recon_loss_pre = jnp.mean((z_pre - x) ** 2)
        recon_loss_all = jnp.mean((z - x) ** 2)
        recon_loss = recon_loss_pre + recon_loss_all
        loss = 0.5 * (recon_loss + bsq_loss)
        return loss

    def __call__(self, x, collect_metrics=True):
        z = self.embed(x)
        for layer in self.encoder:
            z = layer(z)
        z = self.quant_embed(z)

        bsq_loss, quantized, z_indices, metrics = self.tokenizer(z, collect_metrics=collect_metrics)

        quantized_pre = quantized[:, :, :self.s1_bits]
        z_pre = self.post_quant_embed_pre(quantized_pre)
        z = self.post_quant_embed(quantized)

        for layer in self.decoder:
            z_pre = layer(z_pre)
        z_pre = self.head(z_pre)

        for layer in self.decoder:
            z = layer(z)
        z = self.head(z)
        
        loss = self.compute_loss(x, z_pre, z, bsq_loss)
        return loss, quantized, z_indices, metrics



class Kronos(nnx.Module):
    def __init__(self, config, rngs=nnx.Rngs(0)):
        self.s1_bits = config["s1_bits"]
        self.s2_bits = config["s2_bits"]
        self.n_layers = config["n_layers"]
        self.d_model = config["d_model"]
        self.n_heads = config["n_heads"]
        self.ff_dim = config["ff_dim"]
        self.lora_rank = config["lora_rank"]
        self.ffn_dropout_p = config["ffn_dropout_p"]
        self.attn_dropout_p = config["attn_dropout_p"]
        self.resid_dropout_p = config["resid_dropout_p"]
        self.token_dropout_p = config["token_dropout_p"]
        self.s1_vocab_size = 2 ** self.s1_bits
        self.rngs = rngs

        self.token_drop = nnx.Dropout(self.token_dropout_p, rngs=rngs)
        self.embedding = HierarchicalEmbedding(self.s1_bits, self.s2_bits, self.d_model, rngs=rngs)
        self.time_emb = TemporalEmbedding(self.d_model, rngs=rngs)
        self.transformer = nnx.List([
            TransformerBlock(self.d_model, self.n_heads, self.ff_dim, self.ffn_dropout_p, self.attn_dropout_p, self.resid_dropout_p, lora_rank=self.lora_rank, rngs=rngs)
            for _ in range(self.n_layers)
        ])
        self.norm = nnx.RMSNorm(self.d_model, epsilon=1e-5, rngs=rngs)
        self.dep_layer = DependencyAwareLayer(self.d_model, lora_rank=self.lora_rank, rngs=rngs)
        self.head = DualHead(self.s1_bits, self.s2_bits, self.d_model, rngs=rngs)

    def compute_loss(self, s1_logits, s2_logits, s1_targets, s2_targets, padding_mask=None):
        if padding_mask is not None:
            valid = (padding_mask == 0)
            ce_s1 = optax.softmax_cross_entropy_with_integer_labels(s1_logits[valid], s1_targets[valid]).mean()
            ce_s2 = optax.softmax_cross_entropy_with_integer_labels(s2_logits[valid], s2_targets[valid]).mean()
        else:
            ce_s1 = optax.softmax_cross_entropy_with_integer_labels(einshape('bsv->(bs)v', s1_logits), einshape('bs->(bs)', s1_targets)).mean()
            ce_s2 = optax.softmax_cross_entropy_with_integer_labels(einshape('bsv->(bs)v', s2_logits), einshape('bs->(bs)', s2_targets)).mean()
        return (ce_s1 + ce_s2) / 2.0, ce_s1, ce_s2

    def __call__(self, s1_ids, s2_ids, stamp=None, padding_mask=None, s1_targets=None, s2_targets=None):
        x = self.embedding([s1_ids, s2_ids])
        if stamp is not None:
            time_embedding = self.time_emb(stamp)
            x = x + time_embedding
        x = self.token_drop(x)

        for layer in self.transformer:
            x = layer(x, key_padding_mask=padding_mask)

        x = self.norm(x)

        s1_logits = self.head(x)
        sample_s1_ids = self.rngs.categorical(s1_logits, axis=-1)
        sibling_embed = self.embedding.emb_s1(sample_s1_ids)
        x2 = self.dep_layer(x, sibling_embed, key_padding_mask=padding_mask)
        s2_logits = self.head.cond_forward(x2)

        if s1_targets is not None and s2_targets is not None:
            loss, _, _ = self.compute_loss(s1_logits, s2_logits, s1_targets, s2_targets, padding_mask=padding_mask)
            return loss

        return s1_logits, s2_logits, x


TOKENIZER_BASE_CONFIG = {
    "s1_bits": 10,
    "s2_bits": 10,
    "d_in": 6,
    "d_model": 256,
    "ff_dim": 512,
    "n_dec_layers": 4,
    "n_enc_layers": 4,
    "n_heads": 4,
    "attn_dropout_p": 0.0,
    "ffn_dropout_p": 0.0,
    "resid_dropout_p": 0.0,
    "beta": 0.05,
    "zeta": 0.05,
    "gamma": 1.1,
    "gamma0": 1.0,
    "group_size": 4,
    "lora_rank": 0
}

KRONOS_BASE_CONFIG = {
    "s1_bits": 10,
    "s2_bits": 10,
    "d_model": 832,
    "ff_dim": 2048,
    "n_heads": 16,
    "n_layers": 12,
    "attn_dropout_p": 0.0,
    "ffn_dropout_p": 0.2,
    "resid_dropout_p": 0.2,
    "token_dropout_p": 0.0,
    "lora_rank": 0
}

KRONOS_SMALL_CONFIG = {
    "s1_bits": 10,
    "s2_bits": 10,
    "d_model": 512,
    "ff_dim": 1024,
    "n_heads": 8,
    "n_layers": 8,
    "attn_dropout_p": 0.1,
    "ffn_dropout_p": 0.25,
    "resid_dropout_p": 0.25,
    "token_dropout_p": 0.1,
    "lora_rank": 0
}

KRONOS_MINI_CONFIG = {
    "s1_bits": 10,
    "s2_bits": 10,
    "d_model": 256,
    "ff_dim": 512,
    "n_heads": 4,
    "n_layers": 4,
    "attn_dropout_p": 0.0,
    "ffn_dropout_p": 0.2,
    "resid_dropout_p": 0.2,
    "token_dropout_p": 0.0,
    "lora_rank": 0
}

rngs = nnx.Rngs(0)
tokenizer = KronosTokenizer(TOKENIZER_BASE_CONFIG, rngs=rngs)
model = Kronos(KRONOS_BASE_CONFIG, rngs=rngs)

tokenizer.eval()
model.eval()


In [ ]:
nnx.display(tokenizer)

In [ ]:
batch = next(iter(val))
print(batch)
print(nnx.tabulate(tokenizer, x=batch["x"]))
s1, s2 = tokenizer.encode(batch["x"], half=True)
print(nnx.tabulate(model, s1_ids=s1, s2_ids=s2, stamp=batch["stamps"], column_kwargs={"overflow": "fold"}))

In [ ]:
def print_model_structures(pt_state_dict, nnx_model):
    """Print PyTorch state dict keys and Flax NNX model parameter paths side by side."""
    print("=" * 90)
    print(" 1. PYTORCH STATE DICT KEYS & SHAPES")
    print("=" * 90)
    for k, v in pt_state_dict.items():
        shape = tuple(v.shape) if hasattr(v, "shape") else "N/A"
        print(f"  {k:<55} | Shape: {shape}")

    print("\n" + "=" * 90)
    print(" 2. FLAX NNX MODEL STATE KEYS & SHAPES")
    print("=" * 90)
    nnx_flat = flatten_dict(nnx.state(nnx_model).to_pure_dict())
    for path_tuple, val in nnx_flat.items():
        dot_path = ".".join(str(p) for p in path_tuple)
        shape = getattr(val, "shape", "N/A")
        print(f"  {dot_path:<55} | Tuple: {str(path_tuple):<20} | Shape: {shape}")
    print("=" * 90 + "\n")


def load_pt_state_dict_into_nnx(pt_state_dict, nnx_model, inspect_first=True, ignore_keys=("num_batches_tracked")):
    if inspect_first:
        print_model_structures(pt_state_dict, nnx_model)

    pt_numpy = {
        k: (v.detach().cpu().numpy() if hasattr(v, "detach") else np.array(v))
        for k, v in pt_state_dict.items()
    }
    current_nnx_state = nnx.state(nnx_model)
    flat_nnx = flatten_dict(current_nnx_state.to_pure_dict())

    new_flat_nnx = {}
    matched_nnx_keys = set()

    for pt_key, arr in pt_numpy.items():
        if any(pt_key.split('.')[-1] == ignored for ignored in ignore_keys):
            continue
        parts = [int(p) if p.isdigit() else p for p in pt_key.split(".")]
        prefix = parts[:-1]
        suffix = parts[-1]
        if suffix == "weight":
            kernel_path = tuple(prefix + ["kernel"])
            scale_path = tuple(prefix + ["scale"])
            emb_path = tuple(prefix + ["embedding"])

            if kernel_path in flat_nnx:
                target_path = kernel_path
                if arr.ndim == 2:
                    arr = arr.T
                elif arr.ndim == 4:
                    arr = np.transpose(arr, (2, 3, 1, 0))
            elif scale_path in flat_nnx:
                target_path = scale_path
            elif emb_path in flat_nnx:
                target_path = emb_path
            else:
                target_path = tuple(parts)
        elif suffix == "bias":
            target_path = tuple(parts)
        else:
            target_path = tuple(parts)

        if target_path in flat_nnx:
            expected_shape = flat_nnx[target_path].shape
            if arr.shape == expected_shape:
                new_flat_nnx[target_path] = jnp.array(arr)
                matched_nnx_keys.add(target_path)
            else:
                print(f"Shape mismatch at {pt_key} -> {target_path}: PT {arr.shape} vs NNX {expected_shape}")
        else:
            print(f"Target path {target_path} not found in NNX model.")

    unmapped_nnx = set(flat_nnx.keys()) - matched_nnx_keys
    unmapped_weights = [p for p in unmapped_nnx if "rngs" not in p]
    if unmapped_weights:
        missing_paths = [".".join(str(p) for p in p) for p in unmapped_weights]
        print(f"Unmapped NNX parameters (e.g. LoRA params): {missing_paths}... ({len(missing_paths)} total)")
    else:
        print("All base weight parameters successfully mapped!")

    new_nested_state = unflatten_dict(new_flat_nnx)
    nnx.update(nnx_model, new_nested_state)
    print("Weight loading completed successfully.")

## Example Use
import sys
from Kronos.model.kronos import Kronos as PyTorchKronos, KronosTokenizer as PyTorchKronosTokenizer
from flax.traverse_util import flatten_dict, unflatten_dict

"""
tok_cfg = TOKENIZER_BASE_CONFIG.copy()
tok_cfg.pop('lora_rank')  
pt_tokenizer = PyTorchKronosTokenizer(**tok_cfg)

pred_cfg = KRONOS_BASE_CONFIG.copy()
pred_cfg.pop('lora_rank')
pt_predictor_model = PyTorchKronos(**pred_cfg, learn_te=True)
"""

pt_tokenizer = PyTorchKronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
pt_model = PyTorchKronos.from_pretrained("NeoQuasar/Kronos-base")
pt_tokenizer.eval()
pt_model.eval()


print("Transferring weights to Flax Tokenizer...")
load_pt_state_dict_into_nnx(pt_tokenizer.state_dict(), tokenizer, inspect_first=True)

print("Transferring weights to Flax Predictor...")
load_pt_state_dict_into_nnx(pt_model.state_dict(), model, inspect_first=True)



In [ ]:
import os

import faiss
import h5py
import numpy as np


class RagDB:
    def __init__(self):
        self.dim = None
        self.future_len = None
        self.lookback = None
        self.index = None
        self.futures = []
        self.tickers = []
        self.series = []

    def add(self, vectors, futures, tickers, series):
        vectors = np.asarray(vectors, dtype=np.float32)
        futures = np.asarray(futures, dtype=np.float32)
        tickers = np.asarray(tickers, dtype=str)
        series = np.asarray(series, dtype=np.float32)

        if self.index is None:
            self.dim = vectors.shape[-1]
            self.future_len = futures.shape[1]
            self.lookback = series.shape[1]
            self.index = faiss.IndexFlatIP(self.dim)
        elif vectors.shape[-1] != self.dim or futures.shape[1] != self.future_len or series.shape[1] != self.lookback:
            raise ValueError("batch shape mismatch with existing store")

        normalized = np.array(vectors, dtype=np.float32, copy=True)
        faiss.normalize_L2(normalized)
        self.index.add(normalized)

        self.futures.extend(futures)
        self.tickers.extend(tickers.tolist())
        self.series.extend(series)

    def search(self, query, k):
        query = np.asarray(query, dtype=np.float32).reshape(1, -1)
        faiss.normalize_L2(query)
        sims, idxs = self.index.search(query, k)
        idxs = idxs[0]
        futures = np.stack([self.futures[i] for i in idxs])
        series = np.stack([self.series[i] for i in idxs])
        tickers = [self.tickers[i] for i in idxs]
        return sims[0], idxs, tickers, futures, series

    def __len__(self):
        return len(self.tickers)

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        faiss.write_index(self.index, os.path.join(path, "index.faiss"))
        with h5py.File(os.path.join(path, "payloads.h5"), "w") as f:
            f.attrs["dim"] = self.dim
            f.attrs["lookback"] = self.lookback
            f.attrs["future_len"] = self.future_len
            f.create_dataset("ticker", data=np.array(self.tickers, dtype="S"), dtype=h5py.string_dtype("utf-8"))
            f.create_dataset("future", data=np.stack(self.futures))
            f.create_dataset("series", data=np.stack(self.series))

    @classmethod
    def load(cls, path):
        with h5py.File(os.path.join(path, "payloads.h5"), "r") as f:
            dim = int(f.attrs["dim"])
            future_len = int(f.attrs["future_len"])
            lookback = int(f.attrs["lookback"])
            tickers = [str(t) for t in f["ticker"][:]]
            futures = f["future"][:]
            series = f["series"][:]

        db = cls()
        db.dim = dim
        db.future_len = future_len
        db.lookback = lookback
        db.index = faiss.read_index(os.path.join(path, "index.faiss"))
        db.futures = [futures[i] for i in range(len(tickers))]
        db.series = [series[i] for i in range(len(tickers))]
        db.tickers = tickers
        return db



In [ ]:
@nnx.jit
def predict_embeddings(model, s1_ids, s2_ids, stamp):
    model.eval()
    _, _, hidden = model(s1_ids, s2_ids, stamp)
    return hidden[:, -1, :]

def build_db(dataset, lookback, path):
    db = RagDB()
    for i, batch in enumerate(dataset):
        #if i > 5: break
        candle, stamp = batch["x"][:,:lookback,:], batch["stamps"][:,:lookback,:]
        futures = batch["x"][:,lookback:,:]
        tickers = batch["symbol"]

        s1_ids, s2_ids = tokenizer.encode(candle, half=True)
        hidden = predict_embeddings(model, s1_ids, s2_ids, stamp)
        hidden = np.asarray(jax.device_get(hidden))
        db.add(hidden, futures, tickers, candle)
        if i % 10 == 0:
            print(f"[build_db] Processed batch {i}")

    db.save(path)


def rag_profit_probability(dataset, db_path, lookback, k=10, tau=0.1):
    db = RagDB.load(db_path)

    positive = 0
    for i, batch in enumerate(dataset):
        if i > 200: break
        x = batch["x"][0]
        stamps = batch["stamps"][0]
        ticker = batch["symbol"][0]

        candle = x[:lookback][None, ...]
        stamp_past = stamps[:lookback][None, ...]
        future = x[lookback:]

        s1_ids, s2_ids = tokenizer.encode(candle, half=True)
        _, _, hidden = model(s1_ids, s2_ids, stamp_past)

        query = np.asarray(hidden[0, -1, :])

        sims, idxs, tickers, futures, series = db.search(query, k)
        #print(sims[49])
        if sims[0] < 0.98: continue
        rets = futures[:, -1, 3] - series[:, -1, 3]
        w = np.exp(sims / tau)
        w /= w.sum()
        pred_move = float((w * rets).sum())

        actual_move = float(future[-1, 3] - candle[0][-1, 3])
        if (pred_move > 0) == (actual_move > 0): positive += 1

        print(f"{ticker}: pred move = {pred_move:+.3f} over {future.shape[0]} bars "
              f"({k} neighbors, top sim {sims[0]:.3f}), actual move: {actual_move:+.3f}")

    print(f"Score: {positive/i}")

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

    
def rag_single_example(dataset, db_path, lookback, k=1, batch_idx=0, sample_idx=0):
    db = RagDB.load(db_path)

    batch = None
    for i, b in enumerate(dataset):
        if i == batch_idx:
            batch = b
            break

    x = batch["x"][sample_idx]
    stamps = batch["stamps"][sample_idx]
    ticker = batch["symbol"][sample_idx]

    candle = x[:lookback][None, ...]
    stamp_past = stamps[:lookback][None, ...]
    future = x[lookback:]
    F = len(future)

    s1_ids, s2_ids = tokenizer.encode(candle, half=True)
    _, _, hidden = model(s1_ids, s2_ids, stamp_past)
    query = np.asarray(hidden[0, -1, :])

    sims, idxs, tickers, futures, series = db.search(query, k)

    def draw_candles(ax, past, fut, title, future_edges=None):
        n_past = len(past)

        def draw(bars, offset, alpha, edge):
            for i in range(len(bars)):
                o, h, l, c = bars[i][0], bars[i][1], bars[i][2], bars[i][3]
                color = "tab:green" if c >= o else "tab:red"
                ax.plot([offset + i, offset + i], [l, h], color=color, lw=0.8, alpha=alpha)
                ax.add_patch(Rectangle((offset + i - 0.3, min(o, c)), 0.6,
                                       max(abs(c - o), 1e-6),
                                       facecolor=color, edgecolor=edge, alpha=alpha))

        draw(past, 0, 1.0, None)
        draw(fut, n_past, 0.55, future_edges)
        ax.axvline(n_past - 0.5, color="gray", linestyle="--", lw=1)
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.set_xlim(-0.5, n_past + len(fut) - 0.5)

    fig, axes = plt.subplots(3, 2, figsize=(16, 13),
                             sharex="col", gridspec_kw={"height_ratios": [2, 2, 1]})

    draw_candles(axes[0, 0], candle[0], future,
                 f"query {ticker} — past + actual future")
    draw_candles(axes[1, 0], series[0], futures[0],
                 f"neighbor {tickers[0]} (sim {sims[0]:.2f}) — past + stored future")

    for row, past, fut, name in [
        (0, candle[0], future, "query"),
        (1, series[0], futures[0], "neighbor"),
    ]:
        full = np.concatenate([past, fut])
        axes[row, 1].plot(full[:, 4], label=f"{name} volume", color="tab:purple", lw=1)
        axes[row, 1].plot(full[:, 5], label=f"{name} amount", color="tab:brown", lw=1)
        axes[row, 1].axvline(len(past) - 0.5, color="gray", linestyle="--", lw=1)
        axes[row, 1].legend(fontsize=8)
        axes[row, 1].grid(alpha=0.3)
        axes[row, 1].set_title(f"{name} volume / amount")

    plt.tight_layout()
    plt.show()

    return sims, idxs, tickers, futures, series

In [ ]:
LOOKBACK = 90
#build_db(train, LOOKBACK, "store")
rag_profit_probability(val, "store", LOOKBACK, k=1)
#rag_single_example(val, "store", LOOKBACK, batch_idx=22000, sample_idx=0)

# Inference Testing

In [ ]:
import torch

for batch in val:

    x_tensor = torch.from_numpy(batch["x"]).float()  
    stamps_tensor = torch.from_numpy(batch["stamps"]).long()  

    lookback_len = 120

    x_past = x_tensor[:, :lookback_len, :]  
    stamps_past = stamps_tensor[:, :lookback_len, :]  

    x_future = x_tensor[:, lookback_len:, :]  
    stamps_full = stamps_tensor  

    with torch.no_grad():
        s1_tokens, s2_tokens = pt_tokenizer.encode(x_past, half=True)
        logits = pt_model(s1_tokens, s2_tokens, stamps_past )

    print("Tokenizer Outputs (s1 shape):", s1_tokens.shape)
    print("Predictor Logits Shape:", logits.shape if hasattr(logits, 'shape') else [l.shape for l in logits])

    break

import matplotlib.pyplot as plt
logits_s1, logits_s2 = logits
pred_s1 = logits_s1.argmax(dim=-1)  # Shape: (50, 120)
pred_s2 = logits_s2.argmax(dim=-1)  # Shape: (50, 120)

with torch.no_grad():
  x_recon = pt_tokenizer.decode([pred_s1, pred_s2], half=True)

sample_idx = 0
symbol = (
    batch["symbol"][sample_idx]
    if isinstance(batch["symbol"], np.ndarray)
    else batch["symbol"][sample_idx]
)

orig_series = x_past[sample_idx].cpu().numpy()  # (120, 6)
recon_series = x_recon[sample_idx].cpu().numpy()  # (120, 6)

feature_names = ["Open", "High", "Low", "Close", "Volume", "Amount"]

fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()

for i in range(6):
  axes[i].plot(orig_series[:, i], label="Ground Truth", color="tab:blue", lw=1.5)
  axes[i].plot(
      recon_series[:, i],
      label="Reconstructed/Predicted",
      color="tab:orange",
      linestyle="--",
      lw=1.5,
  )
  axes[i].set_title(
      f"{symbol} - {feature_names[i] if i < len(feature_names) else f'Feature {i}'}"
  )
  axes[i].grid(True, alpha=0.3)
  axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
for i, batch in enumerate(val):

    x_tensor = batch["x"]
    stamps_tensor = batch["stamps"]

    lookback_len = 120

    x_past = x_tensor[:, :lookback_len, :]  
    stamps_past = stamps_tensor[:, :lookback_len, :]  

    x_future = x_tensor[:, lookback_len:, :]  
    stamps_full = stamps_tensor  

    s1_tokens, s2_tokens = tokenizer.encode(x_past, half=True)
    logits_s1, logits_s2 = model(s1_tokens, s2_tokens, stamps_past)

    pred_s1 = jnp.argmax(logits_s1, axis=-1)
    pred_s2 = jnp.argmax(logits_s2, axis=-1)
    x_recon = tokenizer.decode([pred_s1, pred_s2], half=True)

    sample_idx = 0
    symbol = batch["symbol"][sample_idx]

    orig_series = np.asarray(x_past[sample_idx])
    recon_series = np.asarray(x_recon[sample_idx])

    feature_names = ["Open", "High", "Low", "Close", "Volume", "Amount"]

    fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
    axes = axes.flatten()

    for i in range(6):
        axes[i].plot(orig_series[:, i], label="Ground Truth", color="tab:blue", lw=1.5)
        axes[i].plot(
            recon_series[:, i],
            label="Flax Reconstructed",
            color="tab:orange",
            linestyle="--",
            lw=1.5,
        )
        axes[i].set_title(f"{symbol} - {feature_names[i]}")
        axes[i].grid(True, alpha=0.3)
        axes[i].legend()

    plt.tight_layout()
    plt.show()
    if i > 2:
        break